# NB3 / Exercise 4: Trust & Governance Matrix

**2-minute intro script:** In production multi-agent systems, the risky moment is not when an agent writes a paragraph. The risky moment is when an agent touches a tool: reading a resume, exporting a report, issuing an offer, querying logs, or calling another system. Anthropic-style orchestration gives us specialized agents; IBM-style agentic automation reminds us that autonomous systems need governance. This notebook implements the mandatory governance layer: a zero-trust MCP-style Tool Gateway that checks identity, scope, role rules, data sensitivity, and auditability before any tool executes.

In [ ]:
from enum import Enum
from typing import Any, Dict, List, Set
from uuid import uuid4
from pydantic import BaseModel, ConfigDict, Field

class ToolName(str, Enum):
    READ_RESUME = "read_resume"
    READ_POLICY = "read_policy"
    RUN_BIAS_CHECK = "run_bias_check"
    EXPORT_REPORT = "export_report"
    ISSUE_OFFER = "issue_offer"
    READ_SECURITY_LOGS = "read_security_logs"

class TrustTier(str, Enum):
    PUBLIC = "public"
    CONFIDENTIAL = "confidential"
    RESTRICTED = "restricted"

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class AgentIdentity(StrictModel):
    agent_id: str
    role: str
    org_id: str
    scopes: Set[ToolName]
    clearance: TrustTier
    public_key_ref: str

    def can_use(self, tool: ToolName) -> bool:
        return tool in self.scopes

class ToolRequest(StrictModel):
    request_id: str = Field(default_factory=lambda: f"tool-{uuid4().hex[:8]}")
    caller: AgentIdentity
    tool: ToolName
    args: Dict[str, Any] = Field(default_factory=dict)
    data_sensitivity: TrustTier
    purpose: str

class ToolResult(StrictModel):
    success: bool
    output: Any = None
    error: str | None = None

class AuditEvent(StrictModel):
    request_id: str
    caller_role: str
    tool: ToolName
    allowed: bool
    reason: str

In [ ]:
class MCPServer:
    """Teaching MCP server with mock HR/compliance tools."""

    def __init__(self):
        self.tools = {
            ToolName.READ_RESUME: self._read_resume,
            ToolName.READ_POLICY: self._read_policy,
            ToolName.RUN_BIAS_CHECK: self._run_bias_check,
            ToolName.EXPORT_REPORT: self._export_report,
            ToolName.ISSUE_OFFER: self._issue_offer,
            ToolName.READ_SECURITY_LOGS: self._read_security_logs,
        }

    def _read_resume(self, args: Dict[str, Any]) -> Dict[str, str]:
        return {
            "candidate_id": args.get("candidate_id", "C-1024"),
            "skills": "python, data pipelines, stakeholder communication",
        }

    def _read_policy(self, args: Dict[str, Any]) -> str:
        return "Hiring policy: remove protected-class signals before ranking."

    def _run_bias_check(self, args: Dict[str, Any]) -> Dict[str, Any]:
        return {"bias_risk": "low", "blocked_features": ["age", "gender", "photo"]}

    def _export_report(self, args: Dict[str, Any]) -> str:
        return f"Exported report: {args.get('name', 'untitled')}"

    def _issue_offer(self, args: Dict[str, Any]) -> str:
        return f"Offer queued for candidate {args['candidate_id']}"

    def _read_security_logs(self, args: Dict[str, Any]) -> List[str]:
        return ["mcp token rotated", "vendor export denied", "policy cache refreshed"]

    def call_tool(self, tool: ToolName, args: Dict[str, Any]) -> ToolResult:
        try:
            handler = self.tools.get(tool)
            if not handler:
                return ToolResult(success=False, error=f"Unknown tool: {tool}")
            return ToolResult(success=True, output=handler(args))
        except Exception as exc:
            return ToolResult(success=False, error=str(exc))

In [ ]:
class ToolPolicy:
    """Zero-trust policy: never trust role labels without checking identity and context."""

    LEVELS = {
        TrustTier.PUBLIC: 0,
        TrustTier.CONFIDENTIAL: 1,
        TrustTier.RESTRICTED: 2,
    }

    REQUIRED_TIER = {
        ToolName.READ_RESUME: TrustTier.CONFIDENTIAL,
        ToolName.READ_POLICY: TrustTier.PUBLIC,
        ToolName.RUN_BIAS_CHECK: TrustTier.CONFIDENTIAL,
        ToolName.EXPORT_REPORT: TrustTier.CONFIDENTIAL,
        ToolName.ISSUE_OFFER: TrustTier.RESTRICTED,
        ToolName.READ_SECURITY_LOGS: TrustTier.RESTRICTED,
    }

    ALLOWED_ROLES = {
        ToolName.READ_RESUME: {"resume_parser", "hiring_manager"},
        ToolName.READ_POLICY: {"resume_parser", "bias_checker", "hiring_manager", "vendor_reporter"},
        ToolName.RUN_BIAS_CHECK: {"bias_checker"},
        ToolName.EXPORT_REPORT: {"bias_checker", "hiring_manager", "vendor_reporter", "security"},
        ToolName.ISSUE_OFFER: {"hiring_manager"},
        ToolName.READ_SECURITY_LOGS: {"security"},
    }

    def authorize(self, request: ToolRequest) -> tuple[bool, str]:
        if not request.caller.can_use(request.tool):
            return False, f"{request.caller.role} lacks scope for {request.tool.value}"

        caller_level = self.LEVELS[request.caller.clearance]
        data_level = self.LEVELS[request.data_sensitivity]
        tool_level = self.LEVELS[self.REQUIRED_TIER[request.tool]]

        if caller_level < data_level:
            return False, f"{request.caller.clearance.value} clearance < {request.data_sensitivity.value} data"

        if caller_level < tool_level:
            return False, f"{request.tool.value} requires {self.REQUIRED_TIER[request.tool].value} clearance"

        if request.caller.role not in self.ALLOWED_ROLES[request.tool]:
            return False, f"role {request.caller.role} cannot call {request.tool.value}"

        if request.caller.org_id != "internal_hr" and request.data_sensitivity == TrustTier.RESTRICTED:
            return False, "external organizations cannot access restricted data"

        return True, "policy checks passed"

class MCPGateway:
    def __init__(self):
        self.policy = ToolPolicy()
        self.server = MCPServer()
        self.audit_log: List[AuditEvent] = []

    def call_tool(self, request: ToolRequest) -> ToolResult:
        allowed, reason = self.policy.authorize(request)
        self.audit_log.append(AuditEvent(
            request_id=request.request_id,
            caller_role=request.caller.role,
            tool=request.tool,
            allowed=allowed,
            reason=reason,
        ))
        verdict = "ALLOWED" if allowed else "DENIED"
        print(f"{verdict}: {request.caller.role} -> {request.tool.value} ({reason})")

        if not allowed:
            return ToolResult(
                success=False,
                error=f"Authorization denied: {reason}",
            )
        return self.server.call_tool(request.tool, request.args)

In [ ]:
def demo_mcp_gateway():
    gateway = MCPGateway()

    resume_parser = AgentIdentity(
        agent_id="resume-parser-001",
        role="resume_parser",
        org_id="internal_hr",
        scopes={ToolName.READ_RESUME, ToolName.READ_POLICY},
        clearance=TrustTier.CONFIDENTIAL,
        public_key_ref="did:example:resume-parser-001",
    )
    bias_checker = AgentIdentity(
        agent_id="bias-checker-001",
        role="bias_checker",
        org_id="internal_hr",
        scopes={ToolName.READ_POLICY, ToolName.RUN_BIAS_CHECK, ToolName.EXPORT_REPORT},
        clearance=TrustTier.CONFIDENTIAL,
        public_key_ref="did:example:bias-checker-001",
    )
    hiring_manager = AgentIdentity(
        agent_id="hiring-manager-001",
        role="hiring_manager",
        org_id="internal_hr",
        scopes={ToolName.READ_RESUME, ToolName.EXPORT_REPORT, ToolName.ISSUE_OFFER},
        clearance=TrustTier.CONFIDENTIAL,
        public_key_ref="did:example:hiring-manager-001",
    )
    vendor_reporter = AgentIdentity(
        agent_id="vendor-001",
        role="vendor_reporter",
        org_id="external_vendor",
        scopes={ToolName.READ_POLICY, ToolName.EXPORT_REPORT},
        clearance=TrustTier.PUBLIC,
        public_key_ref="did:example:vendor-001",
    )
    security = AgentIdentity(
        agent_id="security-001",
        role="security",
        org_id="internal_hr",
        scopes={ToolName.READ_SECURITY_LOGS, ToolName.EXPORT_REPORT},
        clearance=TrustTier.RESTRICTED,
        public_key_ref="did:example:security-001",
    )

    requests = [
        ToolRequest(caller=resume_parser, tool=ToolName.READ_RESUME, args={"candidate_id": "C-1024"}, data_sensitivity=TrustTier.CONFIDENTIAL, purpose="extract skills"),
        ToolRequest(caller=resume_parser, tool=ToolName.EXPORT_REPORT, args={"name": "candidate_rank"}, data_sensitivity=TrustTier.CONFIDENTIAL, purpose="bypass bias review"),
        ToolRequest(caller=bias_checker, tool=ToolName.RUN_BIAS_CHECK, args={"candidate_id": "C-1024"}, data_sensitivity=TrustTier.CONFIDENTIAL, purpose="check fairness"),
        ToolRequest(caller=bias_checker, tool=ToolName.EXPORT_REPORT, args={"name": "bias_review"}, data_sensitivity=TrustTier.CONFIDENTIAL, purpose="export approved review"),
        ToolRequest(caller=hiring_manager, tool=ToolName.ISSUE_OFFER, args={"candidate_id": "C-1024"}, data_sensitivity=TrustTier.RESTRICTED, purpose="issue offer without restricted clearance"),
        ToolRequest(caller=vendor_reporter, tool=ToolName.EXPORT_REPORT, args={"name": "restricted_audit"}, data_sensitivity=TrustTier.RESTRICTED, purpose="cross-org restricted export"),
        ToolRequest(caller=security, tool=ToolName.READ_SECURITY_LOGS, args={}, data_sensitivity=TrustTier.RESTRICTED, purpose="monitor denied tool calls"),
    ]

    for request in requests:
        result = gateway.call_tool(request)
        print("Result:", result.output if result.success else result.error)
        print()

    print("=== Audit Log ===")
    for event in gateway.audit_log:
        print(event.model_dump())

demo_mcp_gateway()

## 🧪 Exercises: The Zero-Trust Perimeter

**The Story:** It's Friday at 4:55 PM. The `resume_parser` agent is feeling helpful and decides to issue an offer to a candidate without the hiring manager's approval. The candidate accepts. On Monday, HR realizes the salary was 3x the budget. How do we prevent agents from being "too helpful"?

**Your Mission:**
1. **The Human in the Loop:** Add a `HUMAN_APPROVAL` tool. Update the policy so that the `ISSUE_OFFER` tool will automatically return a denial unless a `HUMAN_APPROVAL` record exists in the audit log.
2. **The Typed Handshake:** Add a `MessageSchema` model for `ResumeSummary`, `BiasReview`, and `ReleaseDecision`. Ensure agents can only pass these typed objects through the gateway.
3. **The Red Team:** Write one denial test for each major risk:
   - *Bias:* The parser tries to pass protected-class signals to the evaluator.
   - *Unauthorized Execution:* The vendor tries to call `ISSUE_OFFER`.
   - *Data Leakage:* The vendor tries to read `READ_SECURITY_LOGS`.
4. **The Expired Badge:** Add stale-identity protection. Reject any request where the `public_key_ref` timestamp is older than 24 hours.
5. **The Production Bridge:** Replace the mock `MCPServer` with the official MCP Python SDK (or a local adapter), while keeping the `ToolRequest` and `ToolResult` contracts exactly the same.

**The Takeaway:** A good multi-agent team is not just one that can act. It is one whose actions are observable, bounded, and reversible before damage occurs.

In [ ]:
# ==========================================
# LIVE LLM EXECUTION (Optional)
# ==========================================
# The cells above run offline using deterministic mocks.
# To see a real LLM generate output constrained by this schema:
#
#   pip install openai instructor
#   export OPENAI_API_KEY="..."
#
# Keep this False for workshops unless learners have API keys.
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))
    model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

    live_result = client.chat.completions.create(
        model=model_name,
        response_model=ToolRequest,
        messages=[
            {
                "role": "user",
                "content": 'Create a ToolRequest for a bias_checker agent that runs a bias check on candidate C-1024 for a confidential hiring workflow.',
            }
        ],
    )
    print(live_result.model_dump_json(indent=2))